# Planner Node — Unit Tests
Step-by-step testing of the Planner Node functions.
Run each cell independently to inspect intermediate outputs and logs.

In [1]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.planner import planner_node, _build_sections_summary
from prompts.planner_prompts import ExtractionPlan, SectionPlan

logger = get_logger(__name__)
logger.info('Imports OK.')

2026-03-18 01:47:34 [INFO] __main__: Imports OK.


In [2]:
# Step 2 — Run Reader Node to prepare state
# The Planner depends on Reader output, so we run the Reader first.
# This is the same state that LangGraph would pass between the two nodes.

reader_state = {
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
}

reader_result = reader_node(reader_state)

assert len(reader_result["parsed_sections"]) > 0, "Reader returned no sections"

logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s) ready for Planner.")

2026-03-18 01:47:34 [INFO] nodes.reader: Reader Node started.
2026-03-18 01:47:34 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-18 01:47:34 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-18 01:47:34 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-18 01:47:34 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-18 01:47:34 [DEBUG] tools.document_tools: Loaded 1 OpenAPI spec file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_spec'.
2026-03-18 01:47:34 [INFO] nodes.reader: Reader Node complete.
2026-03-18 01:47:34 [INFO] __main__: Reader OK — 238 section(s) ready for Planner.


In [3]:
# Step 3 — Test _build_sections_summary()
# Verifies that the helper function formats sections correctly for the prompt.
# Each row must be: [section_id] title | content preview

summary = _build_sections_summary(reader_result["parsed_sections"])

assert isinstance(summary, str) and len(summary) > 0, "Expected non-empty summary string"

lines = summary.splitlines()
assert all('[' in l and ']' in l for l in lines[:5]), "Expected [section_id] format in each line"

logger.info(f"_build_sections_summary OK — {len(lines)} lines generated.")
logger.info(f"Preview (first 3 lines):")
for line in lines[:3]:
    logger.info(f"  {line[:120]}")

2026-03-18 01:47:34 [INFO] __main__: _build_sections_summary OK — 238 lines generated.
2026-03-18 01:47:34 [INFO] __main__: Preview (first 3 lines):
2026-03-18 01:47:34 [INFO] __main__:   [1] 11.1.0 Introduction | This clause provides the stage 2 definitions of Create, Read, Update and Delete (CRUD) operati
2026-03-18 01:47:34 [INFO] __main__:   [4] 11.1.1.1.1 Description | This operation is invoked by MnS consumer to request the MnS producer to create a Managed O
2026-03-18 01:47:34 [INFO] __main__:   [5] 11.1.1.1.2 Input parameters | Parameter Name          S   Information Type / Legal Values                        Com


In [4]:
# Step 4 — Test planner_node() end-to-end
# Makes a real LLM API call. Ensure OPENAI_API_KEY is set in .env.
# The LLM receives section previews and returns a structured ExtractionPlan.

planner_state = {
    **reader_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
}

planner_result = planner_node(planner_state)

assert "extraction_plan" in planner_result
plan = planner_result["extraction_plan"]

assert "document_summary" in plan
assert "sections_to_extract" in plan
assert isinstance(plan["sections_to_extract"], list)
assert len(plan["sections_to_extract"]) > 0, "Planner returned no sections to extract"

logger.info(f"planner_node OK")
logger.info(f"  document_summary    : {plan['document_summary']}")
logger.info(f"  sections_to_extract : {len(plan['sections_to_extract'])} section(s) selected")

2026-03-18 01:47:34 [INFO] nodes.planner: Planner Node started.
2026-03-18 01:47:34 [DEBUG] nodes.planner: Sending 238 section(s) to Planner LLM.
2026-03-18 01:47:34 [DEBUG] openai._base_client: Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Helper-Method': 'chat.completions.parse', 'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-3229af62-e111-494d-bf85-a1ad7f3999f2', 'post_parser': <function Completions.parse.<locals>.parser at 0x7502653d4c20>, 'content': None, 'json_data': {'messages': [{'content': 'You are an expert in 3GPP technical specifications and OpenAPI design.\n\nYour task is to analyze a list of sections from a 3GPP document and produce\nan extraction plan that will guide the next stage of a rule extraction pipeline.\n\nFor each section you receive, decide:\n  1. Whether it is relevant to OpenAPI rule extraction (skip irrelevant sections).\n  2. What its priority is: high, medium, or lo

In [5]:
# Step 5 — Inspect extraction plan
# Reviews the Planner output in detail: priorities, extraction focus, notes.

sections = plan["sections_to_extract"]

high    = [s for s in sections if s["priority"] == "high"]
medium  = [s for s in sections if s["priority"] == "medium"]
low     = [s for s in sections if s["priority"] == "low"]

logger.info(f"Priority breakdown — high: {len(high)}, medium: {len(medium)}, low: {len(low)}")
logger.info("")
logger.info("High priority sections:")
for s in high[:5]:
    logger.info(f"  [{s['section_id']}] {s['title']}")
    logger.info(f"    Focus : {s['extraction_focus']}")
    if s.get('notes'):
        logger.info(f"    Notes : {s['notes']}")

2026-03-18 01:48:10 [INFO] __main__: Priority breakdown — high: 27, medium: 10, low: 0
2026-03-18 01:48:10 [INFO] __main__: 
2026-03-18 01:48:10 [INFO] __main__: High priority sections:
2026-03-18 01:48:10 [INFO] __main__:   [321] 12.1.1.1 Mapping of operations
2026-03-18 01:48:10 [INFO] __main__:     Focus : Extract mappings of IS operations to RESTful HTTP methods and resource URIs, including CRUD operations and their parameters.
2026-03-18 01:48:10 [INFO] __main__:     Notes : Contains key mappings of internal service operations to RESTful HTTP equivalents, essential for defining OpenAPI paths and operations.
2026-03-18 01:48:10 [INFO] __main__:   [323] 12.1.1.1.2 Operation createMOI
2026-03-18 01:48:10 [INFO] __main__:     Focus : Extract detailed mapping of the createMOI operation input parameters to HTTP PUT method and resource representations.
2026-03-18 01:48:10 [INFO] __main__:     Notes : Defines the create operation mapping critical for OpenAPI POST/PUT endpoint definitions.

In [7]:
sections

[{'section_id': '321',
  'title': '12.1.1.1 Mapping of operations',
  'priority': 'high',
  'extraction_focus': 'Extract mappings of IS operations to RESTful HTTP methods and resource URIs, including CRUD operations and their parameters.',
  'notes': 'Contains key mappings of internal service operations to RESTful HTTP equivalents, essential for defining OpenAPI paths and operations.'},
 {'section_id': '323',
  'title': '12.1.1.1.2 Operation createMOI',
  'priority': 'high',
  'extraction_focus': 'Extract detailed mapping of the createMOI operation input parameters to HTTP PUT method and resource representations.',
  'notes': 'Defines the create operation mapping critical for OpenAPI POST/PUT endpoint definitions.'},
 {'section_id': '324',
  'title': '12.1.1.1.3 Operation getMOIAttributes',
  'priority': 'high',
  'extraction_focus': 'Extract mapping of getMOIAttributes operation to HTTP GET method, including input parameters and response schemas.',
  'notes': 'Important for defining G